
Topic: Running Total in SQL

Objective:
Calculate cumulative sales amount over time using SQL window functions.



In [0]:
%sql

DROP TABLE IF EXISTS sales;

CREATE TABLE sales (
    sale_id INT PRIMARY KEY,
    sale_date DATE,
    customer_id INT,
    amount DECIMAL(10, 2)
);


In [0]:
%sql
INSERT INTO sales (sale_id, sale_date, customer_id, amount) VALUES
(1, '2026-01-01', 101, 100.00),
(2, '2026-01-02', 102, 250.00),
(3, '2026-01-02', 101, 400.00),
(4, '2026-01-03', 101, 150.00),
(5, '2026-01-04', 103, 300.00),
(6, '2026-01-05', 102, 200.00),
(7, '2026-01-06', 101, 400.00);



num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql
-- Solution 1: Overall running total by sale date (Range-based Window Frame)

-- No frame specified here, so SQL defaults to RANGE.
-- It groups duplicate dates (like 2026-01-02) together into a single "peer group"
-- and adds their values at the same time, showing a tied total ($750.00).
SELECT
    sale_id,
    sale_date,
    customer_id,
    amount,
    SUM(amount) OVER (
        ORDER BY sale_date
    ) AS running_total
FROM sales
ORDER BY sale_date;



sale_id,sale_date,customer_id,amount,running_total
1,2026-01-01,101,100.00,100.00
2,2026-01-02,102,250.00,750.00
3,2026-01-02,101,400.00,750.00
4,2026-01-03,101,150.00,900.00
5,2026-01-04,103,300.00,1200.00
6,2026-01-05,102,200.00,1400.00
7,2026-01-06,101,400.00,1800.00


In [0]:
%sql
-- Solution : Overall running total by sale date (Row-based Window Frame)



SELECT
    sale_id,
    sale_date,
    customer_id,
    amount,
    SUM(amount) OVER (
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        -- 'ROWS' forces SQL to calculate strictly line-by-line.
        -- It completely ignores duplicate dates and processes rows one at a time,
        -- creating a true step-by-step rolling sum ($350.00 then $750.00).
    ) AS running_total
FROM sales
ORDER BY sale_date;



sale_id,sale_date,customer_id,amount,running_total
1,2026-01-01,101,100.00,100.00
2,2026-01-02,102,250.00,350.00
3,2026-01-02,101,400.00,750.00
4,2026-01-03,101,150.00,900.00
5,2026-01-04,103,300.00,1200.00
6,2026-01-05,102,200.00,1400.00
7,2026-01-06,101,400.00,1800.00


In [0]:
%sql
-- Solution 2: Running total per customer
SELECT
    sale_id,
    sale_date,
    customer_id,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS customer_running_total
FROM sales
ORDER BY customer_id, sale_date;

sale_id,sale_date,customer_id,amount,customer_running_total
1,2026-01-01,101,100.00,100.00
3,2026-01-02,101,400.00,500.00
4,2026-01-03,101,150.00,650.00
7,2026-01-06,101,400.00,1050.00
2,2026-01-02,102,250.00,250.00
6,2026-01-05,102,200.00,450.00
5,2026-01-04,103,300.00,300.00
